# Comparing MLP vs Naive chrY-based Sex Classifiers

This notebook compares the classification accuracy of an MLP against naive classifiers that use chrY average signal values.
Three naive approaches are demonstrated:

1. **Naive thresholding** — unsupervised, sets a fixed threshold on chrY signal
2. **Gaussian Mixture Model (GMM)** — unsupervised, fits two Gaussians
3. **Logistic Regression** — supervised, learns boundary from labels

They are evaluated at matched coverage (same fraction of low-confidence samples excluded) to enable a fair comparison with the MLP.

In [ ]:
# pylint: disable=missing-module-docstring, import-error, redefined-outer-name

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display  # pylint: disable=import-error
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.mixture import GaussianMixture

## Load and prepare data

In [ ]:
paper_dir = Path.home() / "Projects/epiclass/output/paper"
metadata_dir = paper_dir / "data" / "metadata"
chrY_dir = paper_dir / "data" / "chrY"

if not paper_dir.exists():
    raise FileNotFoundError(f"Directory {chrY_dir} does not exist.")

In [ ]:
# Load metadata
recount3_meta_name = "recount.full_info_metadata.freeze1.tsv"
filepath = metadata_dir / "recount3" / recount3_meta_name

meta_df = pd.read_csv(filepath, sep="\t", low_memory=False)
print(meta_df.shape)
display(meta_df.head())

In [ ]:
# Load chrY values
recount3_chrY_filepath = chrY_dir / "recount3" / "mean_chrY_recount3_100kb.tsv"
value_df = pd.read_csv(recount3_chrY_filepath, sep="\t", low_memory=False)
print(value_df.shape)
display(value_df.head())
print(value_df["mean_chrY"].isna().sum())

In [ ]:
value_df = value_df[~value_df["mean_chrY"].isna()]

In [ ]:
# format: sra.base_sums.[dset]_[id].ALL_1kb_all_none.hdf5
value_df["ID"] = (
    value_df["filename"]
    .str.replace("sra.base_sums.", "", regex=False)
    .str.replace(r"\.ALL.*$", "", regex=True)
    .str.split("_")
    .str[-1]
)
if not value_df["ID"].nunique() == value_df.shape[0]:
    raise ValueError("IDs are not unique.")

In [ ]:
merged_df = pd.merge(left=value_df, right=meta_df, on="ID", how="left")
if not merged_df.shape[0] == value_df.shape[0]:
    raise ValueError("Merge resulted in different number of rows.")

In [ ]:
merged_df["expected_sex"].value_counts(dropna=False)

## Compute features and z-scores

In [ ]:
merged_df["mean_chrY/mean_chrX"] = merged_df["mean_chrY"] / merged_df["mean_chrX"]

In [ ]:
Z_EPS = 1e-12  # numerical safety threshold

for name in ["mean_chrY", "mean_chrX", "mean_chrY/mean_chrX"]:
    print(name)

    # 1. Force numeric, replace inf → NaN
    s = pd.to_numeric(merged_df[name], errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)

    n_missing = s.isna().sum()
    print("Missing values:", n_missing)

    # 2. Compute stats on valid values only
    mean = s.mean(skipna=True)
    std = s.std(skipna=True, ddof=1)

    print(f" mean: {mean:.4f}, std: {std:.4f}")

    # 3. Guard against zero / invalid std
    if not np.isfinite(std) or std < Z_EPS:
        print(" WARNING: std is zero or invalid → z-scores set to NaN\n")
        merged_df[f"{name}_zscore"] = np.nan
        continue

    # 4. Compute z-score safely
    z = (s - mean) / std

    merged_df[f"{name}_zscore"] = z
    print()

In [ ]:
# Filter to labeled samples
merged_df_labeled = merged_df[merged_df["expected_sex"].isin(["female", "male"])]
print(f"Labeled samples: {merged_df_labeled.shape[0]}")

N_missing = merged_df_labeled["mean_chrY/mean_chrX_zscore"].isna().sum()
print(f"Missing z-scores in mean_chrY/mean_chrX ratio column: {N_missing}")

## Visualize distributions

In [ ]:
for name in ["mean_chrY_zscore", "mean_chrY/mean_chrX_zscore"]:
    fig = go.Figure()

    N = merged_df_labeled.shape[0]
    print(f"Plotting {name} for {N} samples.")
    sub_df = merged_df_labeled[~merged_df_labeled[name].isna()]
    print(f"Excluded: {N - sub_df.shape[0]} samples with NA values.")

    female_df = sub_df[sub_df["expected_sex"] == "female"]
    male_df = sub_df[sub_df["expected_sex"] == "male"]

    fig.add_trace(
        go.Violin(
            y=female_df[name],
            x=[0 for _ in range(len(female_df))],
            name="Female",
            fillcolor="lightpink",
            line_color="black",
            spanmode="hard",
            points="outliers",
            marker_size=1,
            box_visible=True,
            meanline_visible=True,
        )
    )

    fig.add_trace(
        go.Violin(
            y=male_df[name],
            x=[1 for _ in range(len(male_df))],
            name="Male",
            fillcolor="lightblue",
            line_color="black",
            spanmode="hard",
            points="outliers",
            marker_size=1,
            box_visible=True,
            meanline_visible=True,
        )
    )

    fig.update_layout(
        height=600,
        title_text=f"Distribution of {name} by Expected Sex",
    )

    fig.show()

## Basic Overlap statistics

In [ ]:
for name in ["mean_chrY_zscore", "mean_chrY/mean_chrX_zscore"]:
    print(f"\n{name}")

    # Subset once
    df = merged_df_labeled[["expected_sex", name]].dropna(subset=[name])

    male_vals = df.loc[df["expected_sex"] == "male", name]
    female_vals = df.loc[df["expected_sex"] == "female", name]

    if male_vals.empty or female_vals.empty:
        print("Not enough data to compute statistic.")
        continue

    # Male Q1
    male_q1 = male_vals.quantile(0.25)

    # % of females above male Q1
    pct_female_above = (female_vals > male_q1).mean() * 100

    print(f"{'Male Q1:':<28} {male_q1:>8.3f}")
    print(
        f"{'Females above male Q1:':<28} "
        f"{pct_female_above:>6.1f}% "
        f"({(female_vals > male_q1).sum():>3}/{len(female_vals)})"
    )

## MLP Classifier framework

MLP reference metrics (from `recount3_metrics_per_assay_assay11c-filtered.tsv`):

- Accuracy: 73.4%
- F1: 48.7%
- Low-confidence fraction: ~11.3% (samples below prediction score 0.6)

We match this low-confidence fraction for the naive classifiers.

In [ ]:
MLP_COVERAGE = 0.886881329524799  # fraction of samples above MLP pred score threshold
TARGET_LOW_CONF = 1 - MLP_COVERAGE

print(f"Target low-confidence fraction: {TARGET_LOW_CONF:.3f}")

## Naive classifiers

### Shared evaluation helpers

In [ ]:
FEATURE_COL = "mean_chrY/mean_chrX_zscore"

In [ ]:
@dataclass
class ClassifierResult:
    """Holds predictions and metadata for a naive classifier."""

    name: str
    predictions: pd.Series  # "male", "female", or "low_confidence"
    low_thresh_score: float | None = None
    high_thresh_score: float | None = None
    decision_boundary: float | None = None


def evaluate_classifier(result: ClassifierResult, labeled_df: pd.DataFrame) -> None:
    """Print classification report and coverage stats for a classifier."""
    print(f"\n{'='*60}")
    print(f"Classifier: {result.name}")
    print(f"{'='*60}")

    df = labeled_df.copy()
    df["prediction"] = result.predictions.reindex(df.index)

    counts = df["prediction"].value_counts(dropna=False)
    total = counts.sum()
    print("\nPrediction distribution:")
    for label, count in counts.items():
        print(f"  {label}: {count} ({count/total:.2%})")

    # Filter to confident predictions
    confident = df[df["prediction"].isin(["male", "female"])]
    low_conf_frac = 1 - len(confident) / len(df)
    print(
        f"\nLow-confidence fraction: {low_conf_frac:.2%} (target: {TARGET_LOW_CONF:.2%})"
    )

    if len(confident) == 0:
        print("WARNING: No confident predictions — skipping classification report.")
        return

    print(f"\nClassification report ({len(confident)} samples):")
    print(
        classification_report(
            y_true=confident["expected_sex"], y_pred=confident["prediction"], digits=4
        )
    )


def plot_classifier(
    result: ClassifierResult, scores: pd.Series, title_suffix: str = ""
) -> None:
    """Violin plot of scores with decision boundary and low-conf zone."""
    fig = go.Figure()

    sub = scores.dropna()
    print(f"Plotting {len(sub)} samples ({len(scores) - len(sub)} excluded for NA)")

    fig.add_trace(
        go.Violin(
            y=sub,
            spanmode="hard",
            points=False,
            box_visible=True,
            meanline_visible=True,
        )
    )

    if result.decision_boundary is not None:
        fig.add_hline(
            y=result.decision_boundary,
            line_dash="solid",
            line_color="red",
            annotation_text="Decision boundary",
            annotation_position="top right",
        )

    if result.low_thresh_score is not None:
        fig.add_hline(
            y=result.low_thresh_score,
            line_dash="dash",
            line_color="orange",
            annotation_text="Low-conf lower",
            annotation_position="bottom right",
        )

    if result.high_thresh_score is not None:
        fig.add_hline(
            y=result.high_thresh_score,
            line_dash="dash",
            line_color="orange",
            annotation_text="Low-conf upper",
            annotation_position="top right",
        )

    fig.update_layout(
        height=600,
        title_text=f"{result.name}: {FEATURE_COL} {title_suffix}",
    )
    fig.show()

### Median-based classifier (naive baseline)

**⚠️ Flawed approach — included for comparison only.**

Places a symmetric gap around the median of the overall z-score distribution.
Samples below the gap are classified as female, above as male, and within the gap as low-confidence. 
The gap width is set so that the same fraction of samples (~11.3%) falls inside it as the MLP's low-confidence fraction.

The problem: the gap is centered on the population median, not on the actual male/female decision boundary.
This means the "uncertain" samples are those near the median of the overall distribution, not those near the region of actual class overlap.

In [ ]:
scores_all = merged_df[FEATURE_COL].dropna()

# Symmetric percentile gap around the median
lower_pct = 50 - (TARGET_LOW_CONF * 100) / 2
upper_pct = 50 + (TARGET_LOW_CONF * 100) / 2

low_thresh_median = np.percentile(scores_all, lower_pct)
high_thresh_median = np.percentile(scores_all, upper_pct)
print(f"Median-based gap: [{low_thresh_median:.3f}, {high_thresh_median:.3f}]")

In [ ]:
median_preds = pd.Series(index=scores_all.index, dtype="object")
for idx, score in scores_all.items():
    if score < low_thresh_median:
        median_preds[idx] = "female"
    elif score > high_thresh_median:
        median_preds[idx] = "male"
    else:
        median_preds[idx] = "low_confidence"

median_boundary = (low_thresh_median + high_thresh_median) / 2
median_result = ClassifierResult(
    name="Median-based (⚠️ flawed)",
    predictions=median_preds,
    decision_boundary=median_boundary,
    low_thresh_score=low_thresh_median,
    high_thresh_score=high_thresh_median,
)

In [ ]:
evaluate_classifier(median_result, merged_df_labeled)

In [ ]:
# plot_classifier(median_result, merged_df[FEATURE_COL])

Results

- acc = 85.73%
- F1 = 85.71%
- % of low-conf samples = 9.71%

### GMM classifier (unsupervised)

**Caveat**: GMMs assume Gaussian components and are sensitive to outliers.
If the chrY/chrX distribution has heavy tails, the GMM may fail to find meaningful clusters. Check the fitted means and variances below.

In [ ]:
X_all = scores_all.values.reshape(-1, 1)

gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(X_all)

means = gmm.means_.flatten()
variances = gmm.covariances_.flatten()
female_comp = int(np.argmin(means))
male_comp = int(np.argmax(means))

print(f"Component means:     {means}")
print(f"Component variances: {variances}")
print(f"Female component: {female_comp} (mean={means[female_comp]:.3f})")
print(f"Male component:   {male_comp} (mean={means[male_comp]:.3f})")

In [ ]:
# Check for GMM failure (component collapsed or absurd separation)
if variances.min() < 1e-4 or abs(means[0] - means[1]) > 50:
    print(
        "\n⚠️  WARNING: GMM likely failed — a component collapsed onto outliers.\n"
        "The male/female clusters are not well-described by two Gaussians.\n"
        "Consider using logistic regression instead.\n"
    )

In [ ]:
# Posterior probabilities and confidence
probs_gmm = gmm.predict_proba(X_all)
prob_male_gmm = probs_gmm[:, male_comp]
confidence_gmm = np.max(probs_gmm, axis=1)

# Decision boundary (where prob_male ≈ 0.5)
score_grid = np.linspace(X_all.min(), X_all.max(), 10_000)
prob_grid = gmm.predict_proba(score_grid.reshape(-1, 1))[:, male_comp]
boundary_gmm = score_grid[np.argmin(np.abs(prob_grid - 0.5))]
print(f"Decision boundary: {boundary_gmm:.3f}")

# Confidence threshold to match target low-conf fraction
conf_threshold_gmm = np.percentile(confidence_gmm, TARGET_LOW_CONF * 100)
actual_low_conf = (confidence_gmm < conf_threshold_gmm).mean()
print(f"Confidence threshold: {conf_threshold_gmm:.3f}")
print(f"Actual low-conf fraction: {actual_low_conf:.3f}")

In [ ]:
# Assign predictions
gmm_preds = pd.Series(index=scores_all.index, dtype="object")
for i, (pm, conf) in enumerate(zip(prob_male_gmm, confidence_gmm)):
    idx = scores_all.index[i]
    if conf < conf_threshold_gmm:
        gmm_preds[idx] = "low_confidence"
    elif pm > 0.5:
        gmm_preds[idx] = "male"
    else:
        gmm_preds[idx] = "female"

# Find score bounds of low-confidence zone for plotting
low_conf_scores = scores_all[gmm_preds == "low_confidence"]
gmm_result = ClassifierResult(
    name="GMM (unsupervised)",
    predictions=gmm_preds,
    decision_boundary=boundary_gmm,
    low_thresh_score=low_conf_scores.min() if len(low_conf_scores) > 0 else None,
    high_thresh_score=low_conf_scores.max() if len(low_conf_scores) > 0 else None,
)

In [ ]:
evaluate_classifier(gmm_result, merged_df_labeled)

The male component collapsed to a single point (variance ~1e-6, which is the GMM's floor) at the extreme outlier value of ~283, and the female component absorbed essentially everything else.

So the GMM didn't make a wide male component — it instead "gave up" on modeling the bulk of males and latched onto a few extreme outliers as one component, while lumping all normal males together with females in the other component. This is a classic GMM failure mode with heavy-tailed data.

This confirms that the GMM approach just isn't a good fit for this data. The chrY/chrX distribution clearly isn't well-described by two Gaussians. I'd go with logistic regression — it'll learn the boundary from your labels without needing to model the full distribution shape, and it handles outliers gracefully since it only cares about the log-odds being linear in the feature.

### Logistic Regression classifier (supervised)

Uses labeled samples to learn the decision boundary.
More robust to outliers and directly comparable to the MLP since both are supervised.

In [ ]:
# Train on labeled data
lr_df = merged_df_labeled[[FEATURE_COL, "expected_sex"]].dropna(subset=[FEATURE_COL])
X_lr = lr_df[FEATURE_COL].values.reshape(-1, 1)
y_lr = (lr_df["expected_sex"] == "male").astype(int).values

lr = LogisticRegression(random_state=42)
lr.fit(X_lr, y_lr)

# Decision boundary: where logit = 0 → score = -intercept / coef
lr_boundary = -lr.intercept_[0] / lr.coef_[0, 0]
print(f"Decision boundary: {lr_boundary:.3f}")

In [ ]:
# Predict on ALL samples (including unlabeled)
X_all_lr = scores_all.values.reshape(-1, 1)
probs_lr = lr.predict_proba(X_all_lr)
prob_male_lr = probs_lr[:, 1]
confidence_lr = np.max(probs_lr, axis=1)

# Confidence threshold to match target low-conf fraction
conf_threshold_lr = np.percentile(confidence_lr, TARGET_LOW_CONF * 100)
actual_low_conf_lr = (confidence_lr < conf_threshold_lr).mean()
print(f"Confidence threshold: {conf_threshold_lr:.3f}")
print(f"Actual low-conf fraction: {actual_low_conf_lr:.3f}")

In [ ]:
# Assign predictions
lr_preds = pd.Series(index=scores_all.index, dtype="object")
for i, (pm, conf) in enumerate(zip(prob_male_lr, confidence_lr)):
    idx = scores_all.index[i]
    if conf < conf_threshold_lr:
        lr_preds[idx] = "low_confidence"
    elif pm > 0.5:
        lr_preds[idx] = "male"
    else:
        lr_preds[idx] = "female"

# Score bounds of low-confidence zone
low_conf_scores_lr = scores_all[lr_preds == "low_confidence"]
lr_result = ClassifierResult(
    name="Logistic Regression (supervised)",
    predictions=lr_preds,
    decision_boundary=lr_boundary,
    low_thresh_score=low_conf_scores_lr.min() if len(low_conf_scores_lr) > 0 else None,
    high_thresh_score=low_conf_scores_lr.max() if len(low_conf_scores_lr) > 0 else None,
)

In [ ]:
evaluate_classifier(lr_result, merged_df_labeled)

In [ ]:
print(f"Median of all scores: {scores_all.median():.3f}")
print(f"LR decision boundary: {lr_boundary:.3f}")
print(f"Median-based gap: [{low_thresh_median:.3f}, {high_thresh_median:.3f}]")
print(
    f"LR low-conf zone:  [{low_conf_scores_lr.min():.3f}, {low_conf_scores_lr.max():.3f}]"
)

In [ ]:
disagree = median_preds != lr_preds
print(f"Disagreements: {disagree.sum()} / {len(disagree)}")

comparison = pd.DataFrame(
    {
        "score": scores_all,
        "median_pred": median_preds,
        "lr_pred": lr_preds,
    }
).loc[disagree]

comparison.value_counts(subset=["median_pred", "lr_pred"])